In [5]:
import veloxchem as vlx
!pwd

/home/marwad/Desktop/Amyloid/7Q4B_Type1


In [ ]:
import requests
import numpy as np
import os
import MDAnalysis as mda

path=os.environ.get('PATH')
home=os.environ.get('HOME')
%env PATH $path:$home/envs/echem-env/bin

def analyse_pdb(pdb_file):
    # Open the pdb file
    f = open(pdb_file, 'r')
    # Read the lines of the pdb file
    lines = f.readlines()
    # Close the pdb file
    f.close()
    # Initialize the number of chains
    n_chains = 0
    # Initialize the paired boolean
    paired = True
    
    # Loop over the lines of the pdb file
    for line in lines:
        # Check if the line starts with COMPND
        if line.startswith('COMPND'):
            # Check if the line contains the word CHAIN
            if 'CHAIN' in line:
                # Get the number of chains from the line
                n_chains = len(line.split('CHAIN:')[1].split(','))
            # Check if the line contains the word MOLECULE
            # Check if the line contains the word PAIRED
            if 'PAIRED' in line:
                # Set the paired boolean to True
                paired = True
    # Return the number of chains, the molecule description and the paired boolean
    return n_chains, paired

def translate(chain, vector, distance):
    unit_vector = np.array(vector) / np.linalg.norm(vector)
    translation_vector = unit_vector * distance
    # Loop over the atoms in the chain
    for atom in chain.values():
        # Translate the coordinates of the atom
        atom['x'] = float(atom['x']) + translation_vector[0]
        atom['y'] = float(atom['y']) + translation_vector[1]
        atom['z'] = float(atom['z']) + translation_vector[2]

def rotate_around_center(chain, center, axis, angle):
    # Define the rotation matrix
    angle_rad = angle * np.pi / 180
    axis = axis / np.linalg.norm(axis)
    a = np.cos(angle_rad / 2)
    b, c, d = axis * np.sin(angle_rad / 2)
    rotation_matrix = np.array([[a*a+b*b-c*c-d*d, 2*(b*c-a*d), 2*(b*d+a*c)],
                                [2*(b*c+a*d), a*a+c*c-b*b-d*d, 2*(c*d-a*b)],
                                [2*(b*d-a*c), 2*(c*d+a*b), a*a+d*d-b*b-c*c]])
    # Loop through the atoms in the chain and rotate them around the center by the angle
    for atom in chain.values():
        coords = np.array([float(atom['x']), float(atom['y']), float(atom['z'])])
        coords -= center
        coords = np.dot(rotation_matrix, coords)
        coords += center
        atom['x'] = str(coords[0])
        atom['y'] = str(coords[1])
        atom['z'] = str(coords[2])

def backbone_atoms(atom):
    # Create a dictionary for the backbone atoms
    backbone = {}
    # Loop over the atoms in the dictionary
    for i in range(0, N):
        # Check if the atom_type is CA, C, N or O
        if atom[i]['atom_type'] == 'CA' or atom[i]['atom_type'] == 'C' or atom[i]['atom_type'] == 'N' or atom[i]['atom_type'] == 'O':
            # Add the atom to the backbone dictionary
            backbone[i] = atom[i]
    # Return the backbone dictionary
    return backbone


# Create a dictionary with the masses of the atoms
masses = {'CA': 12.011, 'N': 14.007, 'O': 15.999, 'C': 12.011}
# Create a function to calculate the center of mass of the backbone atoms

def center_of_mass(atoms):
    # Initialize variables for the total mass and center of mass coordinates
    total_mass = 0
    center_of_mass = [0, 0, 0]
    # Loop through the atoms in the dictionary
    for atom in atoms.values():
        # Calculate the mass of the atom
        mass = masses[atom['atom_type']]
        # Add the mass to the total mass
        total_mass += mass
        # Add the coordinates of the atom multiplied by its mass to the center of mass
        center_of_mass[0] += float(atom['x']) * mass
        center_of_mass[1] += float(atom['y']) * mass
        center_of_mass[2] += float(atom['z']) * mass
    # Divide the center of mass coordinates by the total mass to get the average
    center_of_mass[0] /= total_mass
    center_of_mass[1] /= total_mass
    center_of_mass[2] /= total_mass
    # Return the center of mass
    return center_of_mass

# Append my existing dictionary single_chain with the masses of the atoms
# To know the atom we can look at the atom_type key and focus on the first letter of the atom_type
# The first letter is the chemical element of the atom so we know the masses
# We can use the masses dictionary to get the mass of the atom_type
# We can use the atom_type as a key to get the mass from the masses dictionary


# Define the stacking function. It should take as input the number of chains, the twist angle, the rise and the output file name
# IT shpuld add the end create an ouptu file with n chains stacked with a twist of theta degrees and a rise of rise.

def stack(chain, vector, theta, n, rise, output_file):
    # Initialize the output file
    with open(output_file, 'w') as f:
        # Write the title line
        f.write('Generated by stack.ipynb\n')
        # Write the number of atoms
        f.write(str(len(chain) * n) + '\n')
        # Loop over the number of chains
        for i in range(n):
            # Translate the chain by the vector and the rise
            backbone_atoms(chain)
            translate(chain, vector, distance=rise)
            # Rotate the chain by the angle theta
            rotate_around_center(chain, center_of_mass(backbone), vector, theta)
            # Write the chain to the output file
            for atom in chain.values():
                f.write('{:>5}{:<5}{:>5}{:>5}{:>8.3f}{:>8.3f}{:>8.3f}\n'.format(atom['resnumber'], atom['res'], atom['atom_type'], (int(atom['atom_num']) + i * len(chain)) % 100000, float(atom['x']), float(atom['y']), float(atom['z'])))
        # Write the box dimensions
        f.write(' 20.0 20.0 60.0\n')
        # Close the output file
        f.close()



env: PATH=/opt/pdclab/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/marwad/envs/echem-env/bin


In [2]:
gromacs_file = 'unit.gro'
f = open(gromacs_file, 'r')
lines = f.readlines()
f.close()

# Number of atoms
N = int(lines[1])

# Let's create a dictionary for each atom. Here are the keys: resnumber, res, atom_type, atom_num, x, y, z
# And remove the space

single_chain = {}
for i in range(2, N+2):
    single_chain[i-2] = {'resnumber': lines[i][0:5].strip(), 'res': lines[i][5:10].strip(), 'atom_type': lines[i][10:15].strip(), 'atom_num': lines[i][15:20].strip(), 'x': lines[i][20:28].strip(), 'y': lines[i][28:36].strip(), 'z': lines[i][36:44].strip()}

# Filter through the dictionary to get only the a bakcbone atoms with atom_type: CA, C, N, O
# Create a new dictionary with only the backbone atoms

backbone = {}
for i in range(0, N):
    if single_chain[i]['atom_type'] == 'CA' or single_chain[i]['atom_type'] == 'C' or single_chain[i]['atom_type'] == 'N' or single_chain[i]['atom_type'] == 'O':
        backbone[i] = single_chain[i]


In [3]:
CO_vectors = []

for i in range(0, N):
    # Check if the atom_type is C or O
    if single_chain[i]['atom_type'] == 'C':
        # Calculate the CO vector
        CO_vector = [float(single_chain[i+1]['x']) - float(single_chain[i]['x']), float(single_chain[i+1]['y']) - float(single_chain[i]['y']), float(single_chain[i+1]['z']) - float(single_chain[i]['z'])]
        # Append the CO vector to the list of CO vectors
        CO_vectors.append(CO_vector)

# Sort the CO vectors in function of their direction for those compare them 2 by 2.
# And store them in two categories: positive and negative
# We will use the dot product to compare the direction of the vectors

# Initialize the lists of positive and negative CO vectors
positive_CO_vectors = []
negative_CO_vectors = []
# Loop over the CO vectors
for CO_vector in CO_vectors:
    # Calculate the dot product of the CO vector with the z axis
    dot_product = CO_vector[0] * 0 + CO_vector[1] * 0 + CO_vector[2] * 1
    # Check if the dot product is positive or negative
    if dot_product > 0:
        # Append the CO vector to the list of positive CO vectors
        positive_CO_vectors.append(CO_vector)
    else:
        # Append the CO vector to the list of negative CO vectors
        negative_CO_vectors.append(CO_vector)

# Calculate the average CO vector for the two directions
avg_positive_CO_vector = [0, 0, 0]
avg_negative_CO_vector = [0, 0, 0]
# Loop over the positive CO vectors
for CO_vector in positive_CO_vectors:
    # Add the CO vector to the average positive CO vector
    avg_positive_CO_vector[0] += CO_vector[0]
    avg_positive_CO_vector[1] += CO_vector[1]
    avg_positive_CO_vector[2] += CO_vector[2]
# Loop over the negative CO vectors
for CO_vector in negative_CO_vectors:
    # Add the CO vector to the average negative CO vector
    avg_negative_CO_vector[0] += CO_vector[0]
    avg_negative_CO_vector[1] += CO_vector[1]
    avg_negative_CO_vector[2] += CO_vector[2]

print(avg_positive_CO_vector)
print(len(positive_CO_vectors))
print(avg_negative_CO_vector)
print(len(negative_CO_vectors))

[0.017000000000000792, 0.07699999999999996, 3.327999999999999]
31
[-0.05799999999999805, -0.11900000000000022, -3.1269999999999993]
31


In [7]:
# Call the stacking function
stack(single_chain, avg_positive_CO_vector, -3.214 , 112, 0.495 , 'stack.gro')

In [8]:
# center stacked to the box using gromacs
os.system('gmx editconf -f stack.gro -o fibril_non_eq.gro -c -box 10 10 55.45')

Note that major changes are planned in future for editconf, to improve usability and utility.
Read 114824 atoms
Volume: 24000 nm^3, corresponds to roughly 10800000 electrons
No velocities found
    system size :  7.513  8.080 61.113 (nm)
    center      :  5.156  5.708 34.117 (nm)
    box vectors : 20.000 20.000 60.000 (nm)
    box angles  :  90.00  90.00  90.00 (degrees)
    box volume  :24000.00               (nm^3)
    shift       : -0.156 -0.708 -3.737 (nm)
new center      :  5.000  5.000 30.380 (nm)
new box vectors : 10.000 10.000 60.760 (nm)
new box angles  :  90.00  90.00  90.00 (degrees)
new box volume  :6076.00               (nm^3)


cat: write error: Broken pipe
               :-) GROMACS - gmx editconf, 2026.0-conda_forge (-:

Executable:   /home/marwad/envs/echem-env/bin.AVX2_256/gmx
Data prefix:  /home/marwad/envs/echem-env
Working dir:  /home/marwad/Desktop/Amyloid/7Q4B_Type1
Command line:
  gmx editconf -f test.gro -o test_center.gro -c -box 10 10 60.76


GROMACS reminds you: "Gabba Gabba Hey!" (The Ramones)



0